# Feature Engineering — Fussball Vorhersagen
Version 7: Tordifferenz als neues Feature, form_diff entfernt.

In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt

DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'fussball',
    'user': 'fussball',
    'password': 'fussball_pw'
}

conn = psycopg2.connect(**DB_CONFIG)
print('Verbindung erfolgreich!')

## 1. Alle Daten laden

In [ ]:
query = """
    SELECT 
        m.id,
        m.utc_date,
        m.matchday,
        m.season_id,
        s.start_date AS season_start,
        m.home_team_id,
        m.away_team_id,
        t1.name AS home_team,
        t2.name AS away_team,
        m.home_score_fulltime,
        m.away_score_fulltime,
        m.winner
    FROM matches m
    JOIN teams t1 ON m.home_team_id = t1.id
    JOIN teams t2 ON m.away_team_id = t2.id
    JOIN seasons s ON m.season_id = s.id
    WHERE m.status = 'FINISHED'
    ORDER BY m.utc_date ASC
"""

df = pd.read_sql(query, conn)
df['utc_date'] = pd.to_datetime(df['utc_date'])
df['season_start'] = pd.to_datetime(df['season_start'])
df['season'] = df['season_start'].dt.year

print(f'Spiele total geladen: {len(df)}')
print(f'Saisons: {df["season"].unique()}')
df.head()

## 2. Feature-Funktionen

In [ ]:
def get_form(df, team_id, date, n=5):
    """Punkte der letzten n Spiele."""
    vergangene = df[
        ((df['home_team_id'] == team_id) | (df['away_team_id'] == team_id)) &
        (df['utc_date'] < date)
    ].tail(n)
    punkte = 0
    for _, s in vergangene.iterrows():
        if s['home_team_id'] == team_id:
            if s['winner'] == 'HOME_TEAM': punkte += 3
            elif s['winner'] == 'DRAW': punkte += 1
        else:
            if s['winner'] == 'AWAY_TEAM': punkte += 3
            elif s['winner'] == 'DRAW': punkte += 1
    return punkte


def get_home_form(df, team_id, date, n=5):
    """Punkte der letzten n Heimspiele."""
    spiele = df[
        (df['home_team_id'] == team_id) &
        (df['utc_date'] < date)
    ].tail(n)
    punkte = 0
    for _, s in spiele.iterrows():
        if s['winner'] == 'HOME_TEAM': punkte += 3
        elif s['winner'] == 'DRAW': punkte += 1
    return punkte


def get_away_form(df, team_id, date, n=5):
    """Punkte der letzten n Auswärtsspiele."""
    spiele = df[
        (df['away_team_id'] == team_id) &
        (df['utc_date'] < date)
    ].tail(n)
    punkte = 0
    for _, s in spiele.iterrows():
        if s['winner'] == 'AWAY_TEAM': punkte += 3
        elif s['winner'] == 'DRAW': punkte += 1
    return punkte


def get_heimquote(df, team_id, date, n=12):
    """Anteil Heimsiege aus den letzten n Heimspielen."""
    spiele = df[
        (df['home_team_id'] == team_id) &
        (df['utc_date'] < date)
    ].tail(n)
    if len(spiele) == 0:
        return 0.5
    return (spiele['winner'] == 'HOME_TEAM').mean()


def get_win_streak(df, team_id, date):
    """Aktuelle Siegesserie vor dem Datum."""
    vergangene = df[
        ((df['home_team_id'] == team_id) | (df['away_team_id'] == team_id)) &
        (df['utc_date'] < date)
    ].sort_values('utc_date', ascending=False)
    streak = 0
    for _, s in vergangene.iterrows():
        gewonnen = (
            (s['home_team_id'] == team_id and s['winner'] == 'HOME_TEAM') or
            (s['away_team_id'] == team_id and s['winner'] == 'AWAY_TEAM')
        )
        if gewonnen: streak += 1
        else: break
    return streak


def get_goal_diff(df, team_id, date, n=5):
    """
    Durchschnittliche Tordifferenz der letzten n Spiele.
    Positiv = mehr Tore geschossen als kassiert.
    Sagt mehr als nur Punkte — 5:0 Sieg vs 1:0 Sieg.
    """
    vergangene = df[
        ((df['home_team_id'] == team_id) | (df['away_team_id'] == team_id)) &
        (df['utc_date'] < date)
    ].tail(n)

    if len(vergangene) == 0:
        return 0

    diff = 0
    for _, s in vergangene.iterrows():
        if s['home_team_id'] == team_id:
            diff += s['home_score_fulltime'] - s['away_score_fulltime']
        else:
            diff += s['away_score_fulltime'] - s['home_score_fulltime']
    return diff / len(vergangene)


def is_promoted(df, team_id, season_id):
    """Gibt 1 zurück wenn das Team in der Vorsaison nicht gespielt hat."""
    vorherige = df[
        ((df['home_team_id'] == team_id) | (df['away_team_id'] == team_id)) &
        (df['season_id'] < season_id)
    ]
    return 1 if len(vorherige) == 0 else 0


print('Funktionen definiert!')

## 3. Features berechnen

In [ ]:
print('Berechne Features — dauert etwas...')

features = []
for i, (_, row) in enumerate(df.iterrows()):
    if i % 100 == 0:
        print(f'  {i}/{len(df)} verarbeitet...')

    home_form      = get_form(df, row['home_team_id'], row['utc_date'])
    away_form      = get_form(df, row['away_team_id'], row['utc_date'])
    home_form_home = get_home_form(df, row['home_team_id'], row['utc_date'])
    away_form_away = get_away_form(df, row['away_team_id'], row['utc_date'])
    heimquote      = get_heimquote(df, row['home_team_id'], row['utc_date'])
    home_streak    = get_win_streak(df, row['home_team_id'], row['utc_date'])
    away_streak    = get_win_streak(df, row['away_team_id'], row['utc_date'])
    home_goal_diff = get_goal_diff(df, row['home_team_id'], row['utc_date'])
    away_goal_diff = get_goal_diff(df, row['away_team_id'], row['utc_date'])
    home_promoted  = is_promoted(df, row['home_team_id'], row['season_id'])
    away_promoted  = is_promoted(df, row['away_team_id'], row['season_id'])

    features.append({
        'match_id':         row['id'],
        'utc_date':         row['utc_date'],
        'matchday':         row['matchday'],
        'season_id':        row['season_id'],
        'season':           row['season'],
        'home_team':        row['home_team'],
        'away_team':        row['away_team'],
        'home_form':        home_form,
        'away_form':        away_form,
        'home_form_home':   home_form_home,
        'away_form_away':   away_form_away,
        'heimquote':        heimquote,
        'home_streak':      home_streak,
        'away_streak':      away_streak,
        'home_goal_diff':   home_goal_diff,
        'away_goal_diff':   away_goal_diff,
        'is_promoted_home': home_promoted,
        'is_promoted_away': away_promoted,
        'winner':           row['winner']
    })

df_features = pd.DataFrame(features)
print(f'\nFertig! {len(df_features)} Spiele total')

## 4. Bereinigung — nur 2024 + 2025

In [ ]:
vorher = len(df_features)

df_features = df_features[df_features['season'] >= 2024]
print(f'Nach Saison-Filter: {len(df_features)} Spiele')

df_features = df_features[
    ~((df_features['home_form'] == 0) & (df_features['away_form'] == 0))
]
print(f'Nach 0-0 Bereinigung: {len(df_features)} Spiele')

df_features = df_features[
    ~((df_features['home_form_home'] == 0) & (df_features['away_form_away'] == 0))
]
print(f'Nach Heim/Auswärts Bereinigung: {len(df_features)} Spiele')
print(f'Total rausgelöscht: {vorher - len(df_features)}')
print(f'\nAufsteiger Heimteam: {df_features["is_promoted_home"].sum()}')
print(f'Aufsteiger Auswärtsteam: {df_features["is_promoted_away"].sum()}')

## 5. Features anschauen

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

df_features.groupby('winner')['home_form'].mean().plot(kind='bar', ax=axes[0,0], color='#2a78d6')
axes[0,0].set_title('Heimteam-Form')
axes[0,0].tick_params(rotation=0)

df_features.groupby('winner')['home_goal_diff'].mean().plot(kind='bar', ax=axes[0,1], color='#1baf7a')
axes[0,1].set_title('Heimteam Tordifferenz')
axes[0,1].tick_params(rotation=0)

df_features.groupby('winner')['home_form_home'].mean().plot(kind='bar', ax=axes[1,0], color='#e34948')
axes[1,0].set_title('Heimteam-Form (nur Heimspiele)')
axes[1,0].tick_params(rotation=0)

df_features.groupby('winner')['away_goal_diff'].mean().plot(kind='bar', ax=axes[1,1], color='#f5a623')
axes[1,1].set_title('Auswärtsteam Tordifferenz')
axes[1,1].tick_params(rotation=0)

plt.tight_layout()
plt.show()

print('Ergebnis-Verteilung:')
print(df_features['winner'].value_counts())

## 6. Speichern

In [ ]:
df_features.to_csv('data/features.csv', index=False, encoding='utf-8')
print(f'Features gespeichert: {len(df_features)} Spiele')
print(f'Spalten: {df_features.columns.tolist()}')